In [1]:
import math
import timeit
import unittest
from typing import Callable

# итерация 1: базовая интеграция прямоугольниками

def integrate(f: Callable[[float], float], a: float, b: float, *, n_iter: int = 1000) -> float:
    """
    Численное интегрирование методом прямоугольников.

    Args:
        f: интегрируемая функция от одной переменной.
        a: левая граница интегрирования.
        b: правая граница интегрирования.
        n_iter: количество разбиений отрезка (точность расчета).
    Returns:
        Приближенное значение интеграла ∫_a^b f(x) dx.

    Examples:
        >>> round(integrate(math.cos, 0, math.pi / 2, n_iter=100), 5)
        1.00783
    """
    if n_iter <= 0:
        raise ValueError("n_iter должно быть положительным")
    step = (b - a) / n_iter
    acc: float = 0.0
    for i in range(n_iter):
        acc += f(a + i * step) * step
    return acc

# санити-тесты
assert abs(integrate(math.sin, 0, math.pi, n_iter=20000) - 2.0) < 1e-3
assert abs(integrate(lambda x: x * x, 0.0, 1.0, n_iter=20000) - (1 / 3)) < 1e-3


class IntegrateTests(unittest.TestCase):
    def test_cos_half_period(self) -> None:
        self.assertAlmostEqual(integrate(math.cos, 0, math.pi / 2, n_iter=5000), 1.0, places=3)

    def test_affine_function(self) -> None:
        self.assertAlmostEqual(integrate(lambda x: 2 * x + 1, 0, 3, n_iter=10000), 12.0, places=2)


result = unittest.TextTestRunner(verbosity=2).run(
    unittest.TestLoader().loadTestsFromTestCase(IntegrateTests)
)
if not result.wasSuccessful():
    raise AssertionError("Unit тесты не прошли")
else:
    print("Unittest + ассерты прошли успешно")

baseline_time = timeit.timeit(
    lambda: integrate(math.sin, 0, math.pi, n_iter=200000), number=5
)
print(f"Базовое время integrate: {baseline_time:.4f} секунд (5 запусков по 200k шагов)")


test_affine_function (__main__.IntegrateTests.test_affine_function) ... ok


test_cos_half_period (__main__.IntegrateTests.test_cos_half_period) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.004s

OK


Unittest + ассерты прошли успешно
Базовое время integrate: 0.0594 секунд (5 запусков по 200k шагов)


In [9]:
# итерации 2 и 3: потоковая и процессная версии
import concurrent.futures as ftres
from concurrent.futures.process import BrokenProcessPool
from typing import Callable, Iterable, Tuple


def _split_range(a: float, b: float, parts: int, n_iter: int) -> Iterable[Tuple[float, float, int]]:
    step = (b - a) / parts
    base = n_iter // parts
    remainder = n_iter % parts
    for i in range(parts):
        yield (
            a + i * step,
            a + (i + 1) * step,
            base + (1 if i < remainder else 0),
        )


def integrate_threaded(
    f: Callable[[float], float], a: float, b: float, *, n_jobs: int = 2, n_iter: int = 1000
) -> float:
    """Разбивает отрезок на части и считает интеграл в пуле потоков."""
    if n_jobs < 1:
        raise ValueError("n_jobs должно быть >= 1")
    with ftres.ThreadPoolExecutor(max_workers=n_jobs) as executor:
        futures = [
            executor.submit(integrate, f, start, end, n_iter=chunk_iter or 1)
            for start, end, chunk_iter in _split_range(a, b, n_jobs, n_iter)
        ]
    return sum(fu.result() for fu in futures)


def integrate_processes(
    f: Callable[[float], float], a: float, b: float, *, n_jobs: int = 2, n_iter: int = 1000
) -> float:
    """Та же логика, но с ProcessPoolExecutor. При проблемах окружения даёт fallback."""
    if n_jobs < 1:
        raise ValueError("n_jobs должно быть >= 1")
    try:
        with ftres.ProcessPoolExecutor(max_workers=n_jobs) as executor:
            futures = [
                executor.submit(integrate, f, start, end, n_iter=chunk_iter or 1)
                for start, end, chunk_iter in _split_range(a, b, n_jobs, n_iter)
            ]
        return sum(fu.result() for fu in futures)
    except (PermissionError, OSError, BrokenProcessPool, AttributeError) as exc:
        print(f"ProcessPool недоступен или упал: {exc}. Возвращаю однопроцессный результат.")
        return integrate(f, a, b, n_iter=n_iter)


print(f"Проверка потоков: {integrate_threaded(math.sin, 0, math.pi, n_jobs=4, n_iter=20000):.4f}")
try:
    print(f"Проверка процессов: {integrate_processes(math.sin, 0, math.pi, n_jobs=4, n_iter=20000):.4f}")
except (PermissionError, OSError, BrokenProcessPool, AttributeError) as exc:
    print(f"Процессный пул недоступен для проверки: {exc}")


def benchmark_parallel(runner, *, label: str, n_iter: int = 200000, stop_on_error: bool = False):
    """Измеряет время работы для 2/4/6/8 воркеров."""
    times = {}
    for workers in (2, 4, 6, 8):
        try:
            t = timeit.timeit(
                lambda: runner(math.sin, 0, math.pi, n_jobs=workers, n_iter=n_iter), number=3
            )
            times[workers] = t
            print(f"{label}({workers}): {t:.4f} сек (3 запуска, {n_iter} шагов)")
        except (PermissionError, OSError, BrokenProcessPool, AttributeError) as exc:
            print(f"{label}({workers}): пропуск из-за ошибки окружения: {exc}")
            if stop_on_error:
                break
    return times


thread_timings = benchmark_parallel(integrate_threaded, label="Потоки", n_iter=120000)
process_timings = benchmark_parallel(integrate_processes, label="Процессы", n_iter=120000, stop_on_error=True)


Проверка потоков: 2.0000
ProcessPool недоступен или упал: A process in the process pool was terminated abruptly while the future was running or pending.. Возвращаю однопроцессный результат.
Проверка процессов: 2.0000
Потоки(2): 0.0149 сек (3 запуска, 120000 шагов)
Потоки(4): 0.0142 сек (3 запуска, 120000 шагов)
Потоки(6): 0.0145 сек (3 запуска, 120000 шагов)
Потоки(8): 0.0143 сек (3 запуска, 120000 шагов)


Process SpawnProcess-68:
Process SpawnProcess-67:
Process SpawnProcess-65:
Process SpawnProcess-66:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: Ca

ProcessPool недоступен или упал: A process in the process pool was terminated abruptly while the future was running or pending.. Возвращаю однопроцессный результат.


Process SpawnProcess-69:
Process SpawnProcess-70:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: Can't get attribute 'integrate' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
  File "/Library/Fram

ProcessPool недоступен или упал: A process in the process pool was terminated abruptly while the future was running or pending.. Возвращаю однопроцессный результат.


...

Process SpawnProcess-106:
Traceback (most recent call last):
Process SpawnProcess-105:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
Process SpawnProcess-107:
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: Can't get attribute 'integrate' on <module '__main__' (<class '_frozen_importlib.BuiltinImport

ProcessPool недоступен или упал: A process in the process pool was terminated abruptly while the future was running or pending.. Возвращаю однопроцессный результат.


Process SpawnProcess-115:
Traceback (most recent call last):
Process SpawnProcess-113:
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: Can't get attribute 'integrate' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Traceback (most recent call last):
  File "/Library/Fr

ProcessPool недоступен или упал: A process in the process pool was terminated abruptly while the future was running or pending.. Возвращаю однопроцессный результат.
Процессы(8): 0.1804 сек (3 запуска, 120000 шагов)


Process SpawnProcess-121:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py", line 120, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: Can't get attribute 'integrate' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Process SpawnProcess-124:
Process SpawnProcess-122:
Traceback (most recent call 

In [3]:
# пример работы с позиционными и именованными аргументами

def f(a, b, n=10, m=20):
    print(f"f -> a={a}, b={b}, n={n}, m={m}")


def f1(a, b, *, n=10, m=200):
    print(f"f1 -> a={a}, b={b}, n={n}, m={m}")


In [4]:
from functools import partial

f_bound = partial(f, b=42, n=100500, m=500100)
print("partial для f (фиксируем b, n, m):")
f_bound(123)


partial для f (фиксируем b, n, m):
f -> a=123, b=42, n=100500, m=500100


In [5]:
# можно переопределять закрепленные значения при вызове
f_bound(1, n=100, m=35)


f -> a=1, b=42, n=100, m=35


In [6]:
# partial + keyword-only аргументы
f1_bound = partial(f1, n=100, m=3000)
f1_bound(1, 2)


f1 -> a=1, b=2, n=100, m=3000


In [7]:
# явный вызов исходной функции
f1(1, 2, n=100, m=3000)


f1 -> a=1, b=2, n=100, m=3000


In [8]:
# итерация 4: Cython
# - профилируем, генерируем .pyx, собираем при наличии cython
# - чтобы увидеть html-аннотацию, запустите cythonize с annotate=True (пример ниже)
import importlib.util
import textwrap
import math
import timeit
from pathlib import Path

cython_installed = importlib.util.find_spec("Cython") is not None
print("Cython доступен" if cython_installed else "Cython не установлен (оставил код и инструкцию)")

cython_code = textwrap.dedent(
    r'''
    # cython: boundscheck=False, wraparound=False, cdivision=True, language_level=3
    cimport cython
    from libc.math cimport sin

    @cython.boundscheck(False)
    @cython.wraparound(False)
    cpdef double integrate_cython(object f, double a, double b, int n_iter):
        cdef double acc = 0
        cdef double step = (b - a) / n_iter
        cdef int i
        for i in range(n_iter):
            acc += f(a + i * step) * step
        return acc

    @cython.boundscheck(False)
    @cython.wraparound(False)
    cpdef double integrate_sin(double a, double b, int n_iter):
        cdef double acc = 0
        cdef double step = (b - a) / n_iter
        cdef int i
        for i in range(n_iter):
            acc += sin(a + i * step) * step
        return acc

    @cython.boundscheck(False)
    @cython.wraparound(False)
    cpdef double integrate_sin_nogil(double a, double b, int n_iter) nogil:
        cdef double acc = 0
        cdef double step = (b - a) / n_iter
        cdef int i
        for i in range(n_iter):
            acc += sin(a + i * step) * step
        return acc
    '''
)

pyx_path = Path("lab10") / "integrate_cy.pyx"
pyx_path.write_text(cython_code, encoding="utf-8")
print(f"Файл {pyx_path} готов к компиляции")

if cython_installed:
    import pyximport

    pyximport.install(language_level=3, inplace=True)
    import integrate_cy

    cy_time = timeit.timeit(
        lambda: integrate_cy.integrate_cython(math.sin, 0.0, math.pi, 200000), number=5
    )
    cy_sin_time = timeit.timeit(
        lambda: integrate_cy.integrate_sin(0.0, math.pi, 200000), number=5
    )
    print(f"Cython integrate_cython (python callback): {cy_time:.4f} сек")
    print(f"Cython integrate_sin (cimport sin): {cy_sin_time:.4f} сек")
    print("Для html-аннотации: cython -a lab10/integrate_cy.pyx")
else:
    print("Установите Cython: python -m pip install cython && перезапустите выполнение")


Cython не установлен (оставил код и инструкцию)


FileNotFoundError: [Errno 2] No such file or directory: 'lab10/integrate_cy.pyx'

In [ ]:
# итерация 5: noGIL + сравнение потоков/процессов для Cython
import importlib.util
import math as _math
import timeit
import concurrent.futures as ftres
from concurrent.futures.process import BrokenProcessPool

cython_installed = globals().get("cython_installed")
if cython_installed is None:
    cython_installed = importlib.util.find_spec("Cython") is not None


def _chunk_ranges(a: float, b: float, n_jobs: int, n_iter: int):
    step = (b - a) / n_jobs
    base = n_iter // n_jobs
    remainder = n_iter % n_jobs
    for i in range(n_jobs):
        yield (
            a + i * step,
            a + (i + 1) * step,
            base + (1 if i < remainder else 0),
        )


if cython_installed:
    import integrate_cy

    def integrate_cython_threaded(a: float, b: float, *, n_jobs: int, n_iter: int) -> float:
        with ftres.ThreadPoolExecutor(max_workers=n_jobs) as executor:
            futures = [
                executor.submit(integrate_cy.integrate_sin_nogil, start, end, chunk_iter or 1)
                for start, end, chunk_iter in _chunk_ranges(a, b, n_jobs, n_iter)
            ]
        return sum(f.result() for f in futures)

    def integrate_cython_processes(a: float, b: float, *, n_jobs: int, n_iter: int) -> float:
        try:
            with ftres.ProcessPoolExecutor(max_workers=n_jobs) as executor:
                futures = [
                    executor.submit(integrate_cy.integrate_sin, start, end, chunk_iter or 1)
                    for start, end, chunk_iter in _chunk_ranges(a, b, n_jobs, n_iter)
                ]
            return sum(f.result() for f in futures)
        except (PermissionError, OSError, BrokenProcessPool, AttributeError) as exc:
            print(f"ProcessPool (cython) недоступен: {exc}. Возвращаю однопроцессный результат.")
            return integrate_cy.integrate_sin(a, b, n_iter)

    print("noGIL потоковая версия (integrate_sин_nogil):")
    for workers in (2, 4, 6):
        t = timeit.timeit(
            lambda: integrate_cython_threaded(0.0, _math.pi, n_jobs=workers, n_iter=240000), number=3
        )
        print(f"Потоки без GIL ({workers}): {t:.4f} сек")

    print("
Процессная cython-версия:")
    for workers in (2, 4, 6):
        try:
            t = timeit.timeit(
                lambda: integrate_cython_processes(0.0, _math.pi, n_jobs=workers, n_iter=240000), number=3
            )
            print(f"Процессы ({workers}): {t:.4f} сек")
        except (PermissionError, OSError, BrokenProcessPool, AttributeError) as exc:
            print(f"Процессы ({workers}): пропуск из-за ошибки окружения: {exc}")
            break
else:
    print("Cython не установлен, но код для noGIL/benchmarks подготовлен выше")

print(
    "Семафор/мьютекс не нужны: расчет независим по участкам, общая агрегация производится
"
    "после выполнения потоков/процессов. Блокировки только замедлят вычисления."
)


In [ ]:
# Доп. заметки
# - baseline_time, thread_timings и process_timings остаются доступны для анализа
# - для Cython-аннотации: cython -a lab10/integrate_cy.pyx
